#### FDA Orange Book Data Cleaning

Processe and clean FDA Orange Book data to prepare it for integration with FDA shortage data

#### Objectives

- Clean and standardize data
- Identify relevant features for shortage prediction
- Prepare dataset for merging

In [1]:
#importing all requirements

import pandas as pd

c:\Users\hansi\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
#reading the data

df_orange = pd.read_csv("products.txt", sep='~')

#### Preview dataset

In [3]:
df_orange.head()

,Ingredient,DF;Route,Trade_Name,Applicant,Strength,Appl_Type,Appl_No,Product_No,TE_Code,Approval_Date,RLD,RS,Type,Applicant_Full_Name
0,BUDESONIDE,"AEROSOL, FOAM;RECTAL",BUDESONIDE,PADAGIS ISRAEL,2MG/ACTUATION,A,215328,1,AB,"Apr 12, 2023",No,Yes,RX,PADAGIS ISRAEL PHARMACEUTICALS LTD
1,BUDESONIDE,"AEROSOL, FOAM;RECTAL",UCERIS,SALIX,2MG/ACTUATION,N,205613,1,AB,"Oct 7, 2014",Yes,No,RX,SALIX PHARMACEUTICALS INC
2,MINOCYCLINE HYDROCHLORIDE,"AEROSOL, FOAM;TOPICAL",AMZEEQ,JOURNEY,EQ 4% BASE,N,212379,1,NaN,"Oct 18, 2019",Yes,Yes,RX,JOURNEY MEDICAL CORP
3,AZELAIC ACID,"AEROSOL, FOAM;TOPICAL",AZELAIC ACID,AUROBINDO PHARMA USA,15%,A,210928,1,NaN,"Oct 7, 2020",No,No,DISCN,AUROBINDO PHARMA USA INC
4,BETAMETHASONE VALERATE,"AEROSOL, FOAM;TOPICAL",BETAMETHASONE VALERATE,ALEMBIC,0.12%,A,215832,1,AB,"Aug 22, 2024",No,No,RX,ALEMBIC PHARMACEUTICALS LTD


In [4]:
df_orange.columns

Index(['Ingredient', 'DF;Route', 'Trade_Name', 'Applicant', 'Strength',
       'Appl_Type', 'Appl_No', 'Product_No', 'TE_Code', 'Approval_Date', 'RLD',
       'RS', 'Type', 'Applicant_Full_Name'],
      dtype='object')

In [5]:
df_orange.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47926 entries, 0 to 47925
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Ingredient           47926 non-null  object
 1   DF;Route             47926 non-null  object
 2   Trade_Name           47926 non-null  object
 3   Applicant            47926 non-null  object
 4   Strength             47861 non-null  object
 5   Appl_Type            47926 non-null  object
 6   Appl_No              47926 non-null  int64 
 7   Product_No           47926 non-null  int64 
 8   TE_Code              21629 non-null  object
 9   Approval_Date        47926 non-null  object
 10  RLD                  47926 non-null  object
 11  RS                   47926 non-null  object
 12  Type                 47926 non-null  object
 13  Applicant_Full_Name  47926 non-null  object
dtypes: int64(2), object(12)
memory usage: 5.1+ MB


In [6]:
df_orange.shape

(47926, 14)

In [7]:
#application Number is not unique
#one appl can have multiple drugs 
  
df_orange['Appl_No'].nunique()

26990

In [8]:
df_orange['Product_No'].nunique()

53

#### Create Drug-Level Key

There is no single unique drug identifier for merging.  

Create a standardized drug identifier:
- ingredient (active compound)
- dosage Form + route

In [9]:
#appl no + product no is unique

df_orange[['Appl_No', 'Product_No']].duplicated().sum()

0

In [10]:
#creating primary key with ingredient + df route

df_orange['drug_key'] = (
    df_orange['Ingredient'].str.upper().str.strip() + " | " +
    df_orange['DF;Route'].str.upper().str.strip()
)

#### Feature engineering

Compute the number of unique manufacturers per drug, as a proxy for supply side features

##### Rationale

- More manufacturers → greater supply resilience  
- Fewer manufacturers → higher risk of disruption 

In [11]:
#creating manu count = unique manufactureres per drug

manu_count = (
    df_orange.groupby('drug_key')['Applicant_Full_Name']
    .nunique()
    .reset_index()
    .rename(columns={'Applicant_Full_Name': 'true_manufacturer_count'})
)

In [12]:
manu_count.head()

,drug_key,true_manufacturer_count
0,ABACAVIR SULFATE | SOLUTION;ORAL,3
1,ABACAVIR SULFATE | TABLET;ORAL,7
2,ABACAVIR SULFATE; DOLUTEGRAVIR SODIUM; LAMIVUD...,1
3,ABACAVIR SULFATE; DOLUTEGRAVIR SODIUM; LAMIVUD...,1
4,"ABACAVIR SULFATE; LAMIVUDINE | TABLET, FOR SUS...",1


In [13]:
manu_count.describe()

,true_manufacturer_count
count,4489.000000
mean,4.923145
std,6.234651
min,1.000000
25%,1.000000
50%,2.000000
75%,7.000000
max,59.000000


In [14]:
manu_count.sort_values('true_manufacturer_count', ascending=False).head(10)

,drug_key,true_manufacturer_count
3557,PREDNISONE | TABLET;ORAL,59
2282,IBUPROFEN | TABLET;ORAL,53
44,ACETAMINOPHEN; HYDROCODONE BITARTRATE | TABLET...,52
2796,"METFORMIN HYDROCHLORIDE | TABLET, EXTENDED REL...",47
2169,HYDROCHLOROTHIAZIDE | TABLET;ORAL,46
2764,MEPROBAMATE | TABLET;ORAL,41
1888,FLUDEOXYGLUCOSE F-18 | INJECTABLE;INTRAVENOUS,41
2578,LIDOCAINE HYDROCHLORIDE | INJECTABLE;INJECTION,40
2834,METHOCARBAMOL | TABLET;ORAL,39
953,CHLORTHALIDONE | TABLET;ORAL,39


In [ ]:
#high risk drugs = those with only 1 manufacturer

manu_count[manu_count['true_manufacturer_count'] == 1].head(10)

,drug_key,true_manufacturer_count
2,ABACAVIR SULFATE; DOLUTEGRAVIR SODIUM; LAMIVUD...,1
3,ABACAVIR SULFATE; DOLUTEGRAVIR SODIUM; LAMIVUD...,1
4,"ABACAVIR SULFATE; LAMIVUDINE | TABLET, FOR SUS...",1
7,ABALOPARATIDE | SOLUTION;SUBCUTANEOUS,1
8,ABAMETAPIR | LOTION;TOPICAL,1
9,ABARELIX | POWDER;INTRAMUSCULAR,1
10,ABEMACICLIB | TABLET;ORAL,1
12,ABIRATERONE ACETATE; NIRAPARIB TOSYLATE | TABL...,1
13,ABROCITINIB | TABLET;ORAL,1
14,ACALABRUTINIB MALEATE | TABLET;ORAL,1


In [16]:
#removing discontinued drugs 
#no longer supplied
#including them may distort current supply risk signals


df_orange = df_orange[df_orange['Type'] != 'DISCN']

#### Cleaning and merging with shortage data

- Ingredient names standardized
- Formatting inconsistencies removed
- Multi-ingredient drugs normalized

In [17]:
#clean ingredient names for matching

df_orange['ingredient_clean'] = (
    df_orange['Ingredient']
    .str.upper()
    .str.strip()
    .str.replace(r'\s*;\s*', ';', regex=True)
)

In [18]:
#computing manu counts 
manufacturer_counts = (
    df_orange.groupby('ingredient_clean')['Applicant_Full_Name']
    .nunique()
    .reset_index()
    .rename(columns={'Applicant_Full_Name': 'true_manufacturer_count'})
    )

In [19]:
manufacturer_counts.head()

,ingredient_clean,true_manufacturer_count
0,ABACAVIR SULFATE,6
1,ABACAVIR SULFATE;DOLUTEGRAVIR SODIUM;LAMIVUDINE,1
2,ABACAVIR SULFATE;LAMIVUDINE,5
3,ABALOPARATIDE,1
4,ABEMACICLIB,1


In [20]:
manufacturer_counts.shape

(1881, 2)

In [21]:
manufacturer_counts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1881 entries, 0 to 1880
Data columns (total 2 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   ingredient_clean         1881 non-null   object
 1   true_manufacturer_count  1881 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 29.5+ KB


In [22]:
manufacturer_counts['true_manufacturer_count'].describe()

count    1881.000000
mean        5.641680
std         6.586919
min         1.000000
25%         1.000000
50%         3.000000
75%         8.000000
max        55.000000
Name: true_manufacturer_count, dtype: float64

In [23]:
#storing df as csv

manufacturer_counts.to_csv('manufacturer_counts.csv', index=False)

##### Output

This dataset contains:
- ingredient_clean
- true_manufacturer_count

##### Limitations

- Manufacturer count is a simplified proxy and does not capture:
  - Production capacity differences
  - Supplier dependencies or networks
- Ingredient matching across datasets may introduce inconsistencies
- No demand-side or temporal supply data included at this stage